<a href="https://colab.research.google.com/github/VDJ2824/glaucoma_detection/blob/main/notebooks/03_individual_model_training.ipynb" target="_parent">Open in Colab</a>


# 03 — Independent CNN transfer learning (Phase 3)

Use this notebook in a fresh Colab, Kaggle or local runtime independently of notebooks 00–02. **Training is opt-in and disabled by default.** Existing exploration/preparation notebooks and original images are unchanged. This notebook reads the previously frozen metadata CSV; it never discovers datasets, regenerates splits or accesses test images.

Implementations live in `src/glaucoma/models`, `training` and `utils/reproducibility`. Core models: MobileNetV2, EfficientNetB0 and ResNet50, trained independently. VGG experiments remain separate and are not required. TensorFlow **2.20.0** is CPU-tested on Python 3.12; the recorded Kaggle experiment used **Python 3.13.16, TensorFlow 2.21.0 and Keras 3.13.2**; no `tf_keras`/legacy-Keras switch is used. [TensorFlow transfer-learning guidance](https://www.tensorflow.org/guide/keras/transfer_learning).

## 1. Settings and explicit opt-ins
Choose one model and a saved session from the completed preparation run. Change no frozen split assignments. For a new experiment, point notebook 00's artifact root/run name to fresh storage while referencing the existing frozen CSV. Set `WEIGHTS='imagenet'` and `ALLOW_IMAGENET_DOWNLOAD=True` only when ready for real transfer learning. `weights=None` supports offline CPU construction/smoke tests and is the safe initial default.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import copy

RUN_CLOUD_SETUP = False    # mount Drive and optionally clone/install below
CLONE_REPOSITORY = False
INSTALL_DEPENDENCIES = False
REPO_URL = "https://github.com/VDJ2824/glaucoma_detection.git"
REPO_REF = "main"         # prefer a reviewed tag/commit for reproducible experiments
# Kaggle wins if a hosted image exposes both /kaggle/working and /content.
PLATFORM = "kaggle" if (os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE") or Path("/kaggle/working").is_dir()) else "colab" if (os.environ.get("COLAB_RELEASE_TAG") or Path("/content").is_dir()) else "local"
default_repo = "/content/glaucoma_detection" if PLATFORM == "colab" else "/kaggle/working/glaucoma_detection" if PLATFORM == "kaggle" else str(Path.cwd() if (Path.cwd()/"src").is_dir() else Path.cwd().parent)
REPO_ROOT = Path(os.environ.get("PROJECT_REPO", default_repo))
KAGGLE_INPUT_ROOT = Path(os.environ.get("GLAUCOMA_INPUT_ROOT", "/kaggle/input/your-project-bundle"))  # EDIT to your attachment
if PLATFORM == "kaggle":
    default_session = KAGGLE_INPUT_ROOT / "session.json"
    default_artifacts = Path("/kaggle/working/artifacts")
elif PLATFORM == "colab":
    default_artifacts = Path("/content/drive/MyDrive/glaucoma_detection/artifacts")
    default_session = default_artifacts / "runs/development/environment/session.json"
else:
    default_artifacts = REPO_ROOT / "artifacts"
    default_session = default_artifacts / "runs/development/environment/session.json"
SESSION_FILE = os.environ.get("GLAUCOMA_SESSION", str(default_session))

# Explicit relocation creates derived copies; original split assignments stay intact.
RELOCATE_FROZEN_SESSION = False
SOURCE_METADATA_CSV = KAGGLE_INPUT_ROOT / "acrima_metadata.csv" if PLATFORM == "kaggle" else None
RELOCATED_DATASET_ROOT = KAGGLE_INPUT_ROOT / "database" if PLATFORM == "kaggle" else None
OUTPUT_ARTIFACT_ROOT = Path(os.environ.get("ARTIFACTS_ROOT", str(default_artifacts)))
RUN_NAME = "development"
METADATA_CSV = None        # optionally set to the actual already-saved frozen CSV
MODEL_NAME = "mobilenet_v2"  # efficientnet_b0 or resnet50
WEIGHTS = None             # 'imagenet' for intended transfer learning
ALLOW_IMAGENET_DOWNLOAD = False
LOAD_METADATA = False     # read-only manifest inspection; automatically needed by training
BUILD_MODEL = False
RUN_FORWARD_SMOKE = False
RUN_STAGE1 = False
RUN_STAGE2 = False
RESUME_STAGE1 = False
RESUME_STAGE2 = False
SHOW_HISTORY = False
SAVE_HISTORY_PLOT = False


## 2. Independent Drive mounting, repository and dependency setup
For a fresh Colab runtime, select a GPU if available, set `RUN_CLOUD_SETUP=True` and authorize the Drive mount. Clone only into a nonexistent directory; no automatic pull/reset modifies existing clones. Use `INSTALL_DEPENDENCIES=True` only if needed; installation stops execution for a kernel restart. On the rerun disable installation. Kaggle skips Drive mounting: attach data and configure saved artifacts/session paths explicitly, then save notebook outputs for persistence. CPU also works. No dataset is downloaded by setup.

Python 3.10–3.13 is supported. Requirements preserve TensorFlow 2.20.0/NumPy <2.1 on Python <3.13 and select TensorFlow 2.21.x/NumPy >=2.1 on Python 3.13. Leave installation disabled when the existing hosted environment is already compatible; never downgrade its working GPU stack just to run this notebook. Kaggle detection has priority over `/content`. Colab mounting runs only on Colab.


In [ ]:
if RUN_CLOUD_SETUP:
    if not (3, 10) <= sys.version_info[:2] < (3, 14):
        raise RuntimeError("Use a Python 3.10–3.13 environment for this project.")
    if PLATFORM == "colab":
        from google.colab import drive
        drive.mount("/content/drive")
    if CLONE_REPOSITORY:
        if REPO_ROOT.exists():
            raise FileExistsError("Existing clone preserved; choose another destination or inspect it manually.")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_REF], check=True)
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT/"requirements-training.txt")], check=True, cwd=REPO_ROOT)
        raise RuntimeError("Restart the kernel, disable installation, then rerun this notebook.")
if not (REPO_ROOT/"src/glaucoma").is_dir():
    raise FileNotFoundError("Set REPO_ROOT to the existing clone or enable independent cloud setup.")
sys.path.insert(0, str(REPO_ROOT/"src"))


## 3. Load the saved session and select one core model
The session contains Drive artifact paths and the frozen CSV location. Loading it does not reconfigure the dataset or change stored sessions. A new independent model gets its own checkpoint/history/metric subdirectories. Reusing a model/run directory requires explicit recovery; existing checkpoints are never silently overwritten. Core training cannot select VGG.

The existing normalization is retained: MobileNetV2 [-1,1], ResNet50 BGR with channel means subtracted, EfficientNetB0 float RGB [0,255] for its embedded rescaling. The model factory adds **no preprocessing layer**.

For a Kaggle attachment exported from Drive, set `KAGGLE_INPUT_ROOT` to the directory that directly contains `session.json`, `acrima_metadata.csv` and `database/`. Set `RELOCATE_FROZEN_SESSION=True` once to create a derived session and path-remapped manifest under `/kaggle/working/artifacts/runs/<RUN_NAME>/`. Only `image_path` changes; row order, IDs, labels, patient keys and splits are preserved. Original files are never overwritten. Flat-basename collisions/missing files fail explicitly. Leave `METADATA_CSV=None` to use that derived manifest. Reuse the derived session on later runs without repeating relocation.

Automatic ZIP generation and executed outputs were removed from this reusable notebook. Preserve checkpoints privately through Kaggle saved outputs/downloads. The supplied run's metrics/log/curve are archived separately in [the individual-model experiment records](../docs/experiments/README.md); they are not new results from running these default cells.


In [ ]:
from glaucoma.utils.notebook_runtime import load_session, verify_tensorflow, relocate_frozen_session
from glaucoma.utils.reproducibility import set_random_seed
from glaucoma.models.registry import CORE_APPLICATIONS
from glaucoma.models.factory import build_model, forward_smoke_test
from glaucoma.training.data import load_training_metadata
from glaucoma.training.trainer import train_stage, select_best_checkpoint, recover_best_model, plot_training_history
from glaucoma.training.artifacts import stage_paths

if RELOCATE_FROZEN_SESSION:
    if RELOCATED_DATASET_ROOT is None:
        raise ValueError("Set RELOCATED_DATASET_ROOT to the existing flat image directory.")
    source_csv = SOURCE_METADATA_CSV or load_session(SESSION_FILE)["metadata"]["output"]
    _, relocated_session = relocate_frozen_session(
        SESSION_FILE, source_csv, RELOCATED_DATASET_ROOT, OUTPUT_ARTIFACT_ROOT, RUN_NAME)
    SESSION_FILE = str(relocated_session)
    print("Derived session:", SESSION_FILE)
cfg = copy.deepcopy(load_session(SESSION_FILE))
if MODEL_NAME not in CORE_APPLICATIONS:
    raise ValueError("Choose one of the three core models; optional baselines are separate.")
cfg["model"]["selected"] = MODEL_NAME
cfg["model"]["weights"] = WEIGHTS
if METADATA_CSV is not None:
    cfg["metadata"]["output"] = str(Path(METADATA_CSV))
# Older saved sessions may predate ReduceLROnPlateau configuration; the trainer has the same defaults.
print("Model:", MODEL_NAME, "initialization:", WEIGHTS)
print("Frozen CSV:", cfg["metadata"]["output"])
print("Persistent checkpoints:", stage_paths(cfg, "stage1")["directory"])
print("Both training stages disabled by default; no models or images loaded yet.")


## 4. Inspect the frozen training/validation manifest
Enable `LOAD_METADATA` to inspect without fitting. The shared loader filters out test rows before checking image paths and never opens test images. It requires present binary labels, valid train/validation paths, unique image IDs/paths and known patient isolation. It does not infer missing IDs, recreate splits, deduplicate or augment the saved CSV. Correct invalid manifests in the preparation phase rather than editing them during training.

In [ ]:
metadata = None
if LOAD_METADATA or RUN_STAGE1 or RUN_STAGE2:
    metadata = load_training_metadata(cfg["metadata"]["output"], cfg)
    print(metadata.groupby(["split", "label"]).size().to_string())
    print("Only train/validation metadata retained; CSV unchanged.")
else:
    print("Frozen manifest inspection disabled.")


## 5. Environment, model construction and summary
Enable `BUILD_MODEL=True` for a new Stage 1 model. `WEIGHTS=None` never downloads pretrained parameters. For ImageNet weights, explicitly allow downloading as well. The shared seed helper is called before construction. Architecture: pretrained include_top=False backbone → GAP → Dropout(0.2) → Dense(1, sigmoid). Stage 1 freezes the backbone; its BatchNormalization path is always called with training=False.

In [ ]:
model = None
if BUILD_MODEL:
    print(json.dumps(verify_tensorflow(), indent=2))
    set_random_seed(cfg["seed"])
    if WEIGHTS == "imagenet" and not ALLOW_IMAGENET_DOWNLOAD:
        raise PermissionError("Set ALLOW_IMAGENET_DOWNLOAD=True to request pretrained weights.")
    model = build_model(MODEL_NAME, weights=WEIGHTS)
    model.summary()
else:
    print("Construction disabled. For CPU smoke tests set BUILD_MODEL=True and WEIGHTS=None.")


## 6. Forward-pass smoke test (synthetic only)
This verifies input/output shapes and finite sigmoid output on a synthetic tensor. It does not read images or fit any weights, and its output is not a performance measurement.

In [ ]:
if RUN_FORWARD_SMOKE:
    if model is None:
        raise ValueError("Enable BUILD_MODEL before the forward-pass smoke test.")
    print(forward_smoke_test(model))
else:
    print("Forward-pass smoke test disabled.")


## 7. Stage 1 — frozen backbone, train classification head
Explicitly set `RUN_STAGE1=True` to fit. New training needs a constructed model; epoch-boundary recovery instead uses `RESUME_STAGE1=True` and loads the saved last checkpoint. Adam, binary crossentropy, learning rate 0.001, at most 10 epochs; use the configured early stopping (val_loss), best checkpoint, LR reduction and CSV logging. The unchanged batch utility augments training only and includes the last partial batch.

Checkpoint/history/provenance records are persisted after every completed epoch. Interrupted in-progress batches are not resumable; resume is at the previous committed epoch boundary. Exact augmentation/hardware replay across a restart is not guaranteed. A mismatch in metadata/configuration fails instead of silently mixing experiments.

In [ ]:
stage1_result = None
if RUN_STAGE1:
    if WEIGHTS is None:
        print("Research note: weights=None is random initialization, not ImageNet transfer learning.")
    stage1_result = train_stage(cfg, "stage1", model=model, enabled=True, resume=RESUME_STAGE1)
    print(json.dumps(stage1_result, indent=2))
else:
    print("Stage 1 disabled. No training performed.")


## 8. Stage 2 — restore best Stage 1 and selectively fine-tune
Explicitly set `RUN_STAGE2=True` after Stage 1 completes. The trainer **always reloads best.keras from Stage 1**, regardless of the current in-memory model. Unfreeze the final 20 backbone layers, keep every BatchNormalization layer frozen/inference-only, then recompile with fresh Adam at 0.0001 for at most 20 epochs. `RESUME_STAGE2=True` restores an interrupted Stage 2 last checkpoint and optimizer state instead. Stage 1 settings/frozen manifest must match the recorded contract.

In [ ]:
stage2_result = None
if RUN_STAGE2:
    stage2_result = train_stage(cfg, "stage2", enabled=True, resume=RESUME_STAGE2)
    print(json.dumps(stage2_result, indent=2))
else:
    print("Stage 2 disabled. No fine-tuning performed.")


## 9. Visualize persisted histories
Enable `SHOW_HISTORY` after training or when inspecting a prior saved run. Plots show the actual recorded train/validation loss and accuracy; JSON/CSV histories also include precision, recall, ROC-AUC and learning rates. No plots or scores are fabricated when histories are missing. Set `SAVE_HISTORY_PLOT=True` to save the figure alongside per-model metrics on the configured persistent storage.

In [ ]:
if SHOW_HISTORY:
    import matplotlib.pyplot as plt
    figure = plot_training_history(cfg)
    if SAVE_HISTORY_PLOT:
        output = stage_paths(cfg, "stage1")["selection"].parent / "training_history.png"
        output.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(output, dpi=150)
        print("Saved", output)
    plt.show()
else:
    print("History visualization disabled.")


## 10. Best-checkpoint selection, recovery and persistent outputs
After training, the lower validation loss across Stage 1 and Stage 2 determines selection (ties prefer Stage 1). Fine-tuning can worsen validation loss; Stage 1 remains available. Test images are reserved for a future evaluation notebook. The stored metric snapshot is the logged best validation epoch, not a test evaluation.

The session's Drive run paths receive core/<model>/<stage>/best.keras, last.keras, state.json and contract.json; histories/<model>/<stage>/history.json and epochs.csv; metrics/<model>/<stage>.json and selection.json. Drive must remain mounted and writable. Recovery restores a real full .keras checkpoint; saving paths alone is not a claim of trained performance. Keep run manifests/private paths off Git. In Kaggle, persist/export working outputs explicitly before leaving the session.

In [ ]:
if RUN_STAGE1 or RUN_STAGE2:
    selected = select_best_checkpoint(cfg, save_selection=True)
    print(json.dumps(selected, indent=2))
    # Later explicit recovery, without training:
    # recovered_model = recover_best_model(cfg)
    print("Checkpoints and actual recorded metrics saved using the session's persistent paths.")
else:
    print("No training enabled. No checkpoints, histories or metrics generated.")
